# Train the count-dynamics surrogate on the Luthey-Schulten Minimal Cell trajectories

**What this notebook does**

1. Mounts your Drive copy of `Luthey-Schulten-Lab-Minimal_Cell-db048ac/`.
2. Pulls the `claude/vectorize-gex-propensity-NRqBW` branch of `nikku03/cell`.
3. Probes the 8572 rows of `counts_and_fluxes` to see how many are species counts vs reaction fluxes.
4. One-time converts the 50 replicates from tar+CSV into local parquet so per-epoch I/O drops from minutes to seconds.
5. Trains the count-dynamics MLP defined in `cell_sim/layer_ml/count_dynamics.py` (~18.6 M params, predicts Δlog1p of the 8572-vector).
6. Inspects per-species R² on the held-out replicate and plots a few trajectories.

**Runtime expectations** (on a Colab T4):
- Cells 1–3 (setup + probe): under a minute.
- Cell 4 (parquet conversion): ≈20–40 minutes the first time, near-zero on later runs.
- Cell 5 (training): ≈10 minutes/epoch with parquet cache; tens of minutes/epoch without it.
- Cell 6 (inspection): under a minute.

If anything goes sideways the notebook is safe to re-run from any cell.

## 1. Mount Drive + clone the repo + install requirements

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, subprocess, sys, pathlib

REPO_URL    = 'https://github.com/nikku03/cell.git'
BRANCH      = 'claude/vectorize-gex-propensity-NRqBW'
REPO        = pathlib.Path('/content/cell')
DATASET     = pathlib.Path('/content/drive/MyDrive/Luthey-Schulten-Lab-Minimal_Cell-db048ac')
PARQUET_DIR = pathlib.Path('/content/parquet_cache')
CKPT_DIR    = pathlib.Path('/content/drive/MyDrive/cell_count_dynamics')

if not REPO.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO)])
subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH])
subprocess.check_call(['git', '-C', str(REPO), 'checkout', BRANCH])
subprocess.check_call(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pyarrow', 'matplotlib'])

sys.path.insert(0, str(REPO / 'cell_sim'))
sys.path.insert(0, str(REPO))   # so 'from cell_sim.data import lsdata' resolves
os.environ['LSDATA_ROOT']           = str(DATASET)
os.environ['LSDATA_PARQUET_CACHE']  = str(PARQUET_DIR)
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

import torch
print(f'torch       : {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'repo branch : {subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()}')
print(f'repo HEAD   : {subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"]).decode().strip()}')
print(f'dataset     : {DATASET}  exists={DATASET.exists()}')
print(f'parquet     : {PARQUET_DIR}')
print(f'checkpoints : {CKPT_DIR}')

## 2. Sanity-check the registry + count rows

In [ ]:
from cell_sim.data import lsdata

info = lsdata.check_registry()
print(f'root       : {info["root"]}')
print(f'present    : {len(info["present"])}/27 small assets')
if info['missing']:
    print(f'missing    : {info["missing"]}')

n_species = len(lsdata.replicate_species(1))
rep_indices = lsdata.list_replicates()
print(f'\nreplicate count : {len(rep_indices)} (indices {min(rep_indices)}..{max(rep_indices)})')
print(f'rows / replicate: {n_species}')

## 3. Probe the row composition of `counts_and_fluxes`

We want to know how many of the 8572 rows are species counts (non-negative integers) vs reaction fluxes (signed floats, sometimes NaN). The training transform handles all three cases now, but if fluxes dominate the row count it's worth knowing for interpretation.

In [ ]:
import collections
import numpy as np

species_names = lsdata.replicate_species(1)
prefixes = collections.Counter(s.split('_')[0] for s in species_names)
print('Top 15 row prefixes:')
for p, n in prefixes.most_common(15):
    print(f'  {p:>20s}  {n:>5d}')

df_t100 = lsdata.load_replicate(1, time_start=100.0, time_end=100.0)
vals = df_t100.iloc[:, 0].to_numpy()
print(f'\nAt t=100s in replicate 1:')
print(f'  finite      : {np.isfinite(vals).sum():>5d}')
print(f'  NaN         : {np.isnan(vals).sum():>5d}')
print(f'  negative    : {(vals < 0).sum():>5d}')
print(f'  zero        : {(vals == 0).sum():>5d}')
print(f'  positive    : {(vals > 0).sum():>5d}')
print(f'  min / max   : {np.nanmin(vals):.3f} / {np.nanmax(vals):.3f}')

## 4. One-time parquet conversion

Reading 49 replicates per epoch off Drive dominates wall time — each replicate is a 268 MB tar member. Converting to local parquet (~30–50 MB each) makes per-epoch reads near-instant.

Skip this cell if you already ran it once. The training cell auto-uses the cache via `LSDATA_PARQUET_CACHE` when files are present.

In [ ]:
import time

t0 = time.time()
for i in range(1, 51):
    out = PARQUET_DIR / f'counts_and_fluxes.{i}.parquet'
    if out.exists():
        continue
    t = time.time()
    df = lsdata.load_replicate(i)            # tar stream path
    df.to_parquet(out, compression='zstd')
    print(f'  rep {i:>2d}  {out.stat().st_size/1024**2:>6.1f} MB  '
          f'(read+write {time.time()-t:.1f}s)')
print(f'\nTotal conversion: {time.time()-t0:.0f}s')

total = sum(p.stat().st_size for p in PARQUET_DIR.glob('*.parquet'))
print(f'Cache size: {total/1024**3:.2f} GB across '
      f'{len(list(PARQUET_DIR.glob("*.parquet")))} files')

## 5. Train the count-dynamics surrogate

Defaults: 49 replicates train, 1 hold-out, 2 epochs, batch=256, hidden=1024, 2 residual blocks. ~18.6 M params. Loss is MSE on Δ (signed-log1p of the 8572-vector).

Bump `n_epochs` once you see a healthy first-epoch loss curve.

In [ ]:
from cell_sim.lgnn.training.train_mlp import (
    TrainConfig, train_count_dynamics)

cfg = TrainConfig(
    n_species=n_species,
    hidden=1024, n_blocks=2, dropout=0.0,
    batch_size=256, lr=3e-4, weight_decay=1e-5,
    train_replicates=tuple(range(1, 50)),
    val_replicates=(50,),
    n_epochs=2,
    device='auto',
    log_every=200,
)
ckpt_path = CKPT_DIR / 'count_dynamics_v0.pt'
out = train_count_dynamics(cfg, lsdata, checkpoint_path=ckpt_path)
print('\n=== Training complete ===')
print(f'best val MSE : {out["best_val_mse"]:.4f}')
print(f'history      : {out["history"]}')
print(f'checkpoint   : {ckpt_path}')

## 5b. Build the species reaction graph (week-2 prep)

Parses the iMB155 COBRA JSON and builds a species×species edge index where two rows are connected if they appear in the same reaction. Rows in the 8572-vector that don't match an SBML metabolite ID get a self-loop only — the M1 GNN will fall back to a per-node MLP on those, identical to the M0 MLP behaviour. This cell only needs to run once; it saves the graph to Drive so future training cells can `load_species_graph(...)`.

In [ ]:
from cell_sim.lgnn.data.species_graph import (
    build_species_graph, diagnose_row_matching,
    graph_summary, save_species_graph,
    simulator_edge_summary, flux_edge_summary)

row_names = lsdata.replicate_species(1)

# 1. SBML diagnostic — choose best metabolite-graph source
candidates = []
syn3a_xml = REPO / 'cell_sim' / 'data' / 'Minimal_Cell_ComplexFormation' / 'input_data' / 'Syn3A_updated.xml'
if syn3a_xml.exists():
    candidates.append(('Syn3A_updated.xml', syn3a_xml))
try:
    candidates.append(('iMB155 COBRA JSON', lsdata.get('cobra_imb155')))
except Exception as e:
    print(f'iMB155 not available ({e}), skipping')

best = None
for label, p in candidates:
    print(f'\n=== sbml diagnostic: {label} ===')
    rep = diagnose_row_matching(row_names, p)
    for k in ['n_sbml_species', 'n_total_match', 'matches_per_rule']:
        print(f'  {k:25s}  {rep[k]}')
    if best is None or rep['n_total_match'] > best[1]['n_total_match']:
        best = (label, rep, p)
label, rep, src_path = best

# 2. Simulator-edge diagnostic — central-dogma chain
print(f'\n=== simulator-edge diagnostic ===')
sim_rep = simulator_edge_summary(row_names)
print(f'  n_loci_seen           : {sim_rep["n_loci_seen"]}')
print(f'  n_loci_with_G_R_P_all : {sim_rep["n_loci_with_G_R_P_all"]}')
print(f'  total_simulator_edges : {sim_rep["total_simulator_edges"]}')
print(f'  edges per pattern:')
for label_, n in sim_rep['edges_per_pattern'].items():
    print(f'    {label_:50s}  {n}')
print(f'  sample loci           : {sim_rep["sample_loci"]}')

# 3. Flux-edge diagnostic — F_<rxn_id> coverage
print(f'\n=== flux-edge diagnostic ===')
flux_rep = flux_edge_summary(row_names, src_path)
for k in ('n_sbml_reactions', 'n_with_F_avg_row', 'n_with_F_end_row',
         'n_reactions_covered', 'n_unmatched_reactions'):
    print(f'  {k:25s}  {flux_rep[k]}')
print(f'  sample (rxn, has_avg, has_end):')
for s_ in flux_rep['sample_with_flux_rows']:
    print(f'    {s_}')

# 4. Build the full graph (SBML + simulator + flux + self-loops)
print(f'\nBuilding graph with: {label} (matched {rep["n_total_match"]} rows)\n'
      f'  + simulator edges + flux↔species edges')
g = build_species_graph(row_names, src_path, include_simulator_edges=True)
print('\nGraph stats:')
for k, v in graph_summary(g).items():
    print(f'  {k:30s}  {v}')

graph_path = CKPT_DIR / 'species_graph_full.pt'
save_species_graph(g, graph_path)
print(f'\nsaved -> {graph_path}')


## 5c. Pre-M1 acceptance gate

Before spending a T4-hour on M1 training, the graph and the data loader
have to clear four checks. Each line below evaluates to PASS or FAIL —
if any fails, fix that thing before training M1.

| # | criterion | rationale |
|---|---|---|
| 1 | `n_orphan_nodes` ≤ 4500 (without flux fix) or ≤ 2500 (with flux fix) | the message-passing layers must touch the rows we care about |
| 2 | `n_loci_with_G_R_P_all` ≥ 450 | central-dogma parser caught nearly all loci |
| 3 | every PM_xxxx row has a non-empty incoming edge list from its RP_xxxx | the chain is wired correctly for the M1 thesis rows |
| 4 | M0 with `buffered_shuffle=True, buffer_size=8` is within ±0.05 of the M0a top-100 median R² | loader change doesn't break the baseline before we re-use it for M1 |


In [ ]:
# === Gate 1, 2, 3 — graph-side checks ===
from cell_sim.lgnn.data.species_graph import load_species_graph, EdgeKind
import re

g = load_species_graph(graph_path)
stats = graph_summary(g)
n_orph = stats['n_orphan_nodes']
sim_rep = simulator_edge_summary(g.row_names)
n_loci_full = sim_rep['n_loci_with_G_R_P_all']

# (1) orphan-node budget. Threshold depends on whether flux edges fired.
n_flux = stats['edges_per_kind']['FLUX_COUPLING']
thresh1 = 2500 if n_flux > 0 else 4500
g1 = (n_orph <= thresh1)
print(f'[{"PASS" if g1 else "FAIL"}] gate 1   n_orphan_nodes={n_orph}'
      f'  (threshold {thresh1})  flux_edges={n_flux}')

# (2) locus-coverage budget
g2 = (n_loci_full >= 450)
print(f'[{"PASS" if g2 else "FAIL"}] gate 2   n_loci_with_G_R_P_all={n_loci_full}'
      f'  (threshold 450)')

# (3) PM_xxxx rows must each have ≥1 incoming non-self edge from RP_xxxx
name_to_idx = {n: i for i, n in enumerate(g.row_names)}
src, dst = g.edge_index[0].tolist(), g.edge_index[1].tolist()
kind = g.edge_kind.tolist()
pm_rows = [n for n in g.row_names if re.match(r'^PM_\d{3,}', n)]
incoming_from_rp = {n: 0 for n in pm_rows}
for s_, d_, k_ in zip(src, dst, kind):
    if k_ == int(EdgeKind.SELF_LOOP):
        continue
    dst_name = g.row_names[d_]
    src_name = g.row_names[s_]
    if dst_name in incoming_from_rp and src_name.startswith('RP_'):
        incoming_from_rp[dst_name] += 1
n_pm = len(pm_rows)
n_pm_with_rp = sum(1 for v in incoming_from_rp.values() if v > 0)
g3 = (n_pm_with_rp >= 0.95 * n_pm) if n_pm > 0 else True
print(f'[{"PASS" if g3 else "FAIL"}] gate 3   PM_xxxx with incoming RP_xxxx edges:'
      f' {n_pm_with_rp}/{n_pm}')
if not g3:
    sample_orphan_pm = [n for n, v in incoming_from_rp.items() if v == 0][:5]
    print(f'           sample orphan PM_xxxx (no RP_ → PM_ edge): {sample_orphan_pm}')

print('\n--- gate 4 (loader-change M0 sanity) is a separate train run; '
      'see next cell ---')


In [ ]:
# === Gate 4 — re-run M0 with buffer_size=8, compare to M0a baseline ===
# Skip this cell if you already have the gate-4 measurement; it costs
# the same as one M0 training run (~45 min on T4).
from cell_sim.lgnn.training.train_mlp import TrainConfig, train_count_dynamics
from cell_sim.lgnn.data.dataset import replicate_to_log1p_array
import numpy as np, torch
from cell_sim.lgnn.models.mlp_baseline import CountDynamicsMLP

cfg_g4 = TrainConfig(
    n_species=n_species, hidden=1024, n_blocks=2, dropout=0.0,
    batch_size=256, lr=3e-4, weight_decay=1e-5,
    train_replicates=tuple(range(1, 50)), val_replicates=(50,),
    n_epochs=2, device='auto', log_every=200,
    buffered_shuffle=True, buffer_size=8,
)
ckpt_g4 = CKPT_DIR / 'count_dynamics_v0_buf8.pt'
_ = train_count_dynamics(cfg_g4, lsdata, checkpoint_path=ckpt_g4)

# Compute top-100 high-variance median R² on rep 50
model_g4 = CountDynamicsMLP(n_species=cfg_g4.n_species,
                            hidden=cfg_g4.hidden, n_blocks=cfg_g4.n_blocks)
model_g4.load_state_dict(torch.load(ckpt_g4, weights_only=False)['state_dict'])
model_g4.eval()
df50 = lsdata.load_replicate(50)
X = replicate_to_log1p_array(df50)
DX = X[1:] - X[:-1]
X_in = X[:-1]                      # slice FIRST so P has T-1 rows
with torch.no_grad():
    P = []
    for s in range(0, X_in.shape[0], 512):
        P.append(model_g4(torch.from_numpy(X_in[s:s+512])).numpy())
P = np.concatenate(P, axis=0)
assert P.shape == DX.shape, (P.shape, DX.shape)
ss_res = ((P - DX) ** 2).sum(axis=0)
ss_tot = ((DX - DX.mean(axis=0, keepdims=True)) ** 2).sum(axis=0)
r2_g4 = np.where(ss_tot > 0, 1 - ss_res / np.where(ss_tot > 0, ss_tot, 1), np.nan)
top100 = np.argsort(X.var(axis=0))[::-1][:100]
median_g4 = float(np.nanmedian(r2_g4[top100]))

M0A_MEDIAN = -0.502         # see EXPERIMENTS.md, M0a row
delta = median_g4 - M0A_MEDIAN
g4 = abs(delta) <= 0.05
print(f'\n[{"PASS" if g4 else "FAIL"}] gate 4   M0(buf=8) top-100 median R² = {median_g4:+.3f}'
      f'  (M0a baseline {M0A_MEDIAN:+.3f}, delta {delta:+.3f})')
if g4:
    print('         → safe to use buffered_shuffle=True, buffer_size=8 for M1')
else:
    print('         → loader change degraded M0; investigate before training M1')


## 6. Inspect per-species R² + plot a few trajectories

Mean R² over 8572 rows is hard to interpret because most species are constant or sparse. Look at the top-100 highest-variance rows (where the model has any chance of explaining variance) and plot a couple of trajectories with the predicted `dx` rolled forward from x at t=0.

In [ ]:
import numpy as np
import re
import torch
from cell_sim.lgnn.models.mlp_baseline import CountDynamicsMLP
from cell_sim.lgnn.data.dataset import replicate_to_log1p_array

ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model = CountDynamicsMLP(
    n_species=cfg.n_species, hidden=cfg.hidden, n_blocks=cfg.n_blocks)
model.load_state_dict(ckpt['state_dict'])
model.eval()

df = lsdata.load_replicate(50)                       # held-out
X = replicate_to_log1p_array(df)                     # (T, S) signed-log1p
T = X.shape[0]
DX = X[1:] - X[:-1]                                  # (T-1, S) ground-truth dx

X_in = X[:-1]
with torch.no_grad():
    pred = []
    bs = 512
    for s in range(0, X_in.shape[0], bs):
        x = torch.from_numpy(X_in[s:s+bs])
        pred.append(model(x).numpy())
PRED = np.concatenate(pred, axis=0)                  # (T-1, S)
assert PRED.shape == DX.shape, (PRED.shape, DX.shape)

# Per-species R²
ss_res = ((PRED - DX) ** 2).sum(axis=0)
ss_tot = ((DX - DX.mean(axis=0, keepdims=True)) ** 2).sum(axis=0)
with np.errstate(divide='ignore', invalid='ignore'):
    r2 = 1.0 - ss_res / np.where(ss_tot > 0, ss_tot, 1.0)
r2 = np.where(ss_tot > 0, r2, np.nan)

names = list(df.index)
var_per_row = X.var(axis=0)

# === Headline metric A: top-100 high-variance rows (the M0-era headline) ===
top_idx = np.argsort(var_per_row)[::-1][:100]
print('=== headline A: top-100 high-variance rows ===')
print(f'  median R² : {np.nanmedian(r2[top_idx]):+.3f}')
print(f'  mean R²   : {np.nanmean(r2[top_idx]):+.3f}')
print(f'  >0.5      : {(r2[top_idx] > 0.5).sum()} / 100')
print(f'  >0.0      : {(r2[top_idx] > 0.0).sum()} / 100')

# === Headline metric B: PM_xxxx rows specifically (the M1 thesis test) ===
# This is the metric the graph build was designed to move. M0 hits ~-8 here.
# M1 should ideally beat -1 (graph thesis) and ideally cross 0 (graph wins).
pm_idx = np.array([i for i, n in enumerate(names) if re.match(r'^PM_\d{3,}', n)],
                  dtype=np.int64)
if len(pm_idx) > 0:
    pm_r2 = r2[pm_idx]
    pm_r2_finite = pm_r2[np.isfinite(pm_r2)]
    print(f'\n=== headline B: PM_xxxx rows (M1 thesis target) ===')
    print(f'  count        : {len(pm_idx)} ({np.isfinite(pm_r2).sum()} finite)')
    print(f'  median R²    : {np.nanmedian(pm_r2):+.3f}')
    print(f'  10th pct R²  : {np.nanpercentile(pm_r2, 10):+.3f}')
    print(f'  >0.0         : {(pm_r2_finite > 0).sum()} / {len(pm_r2_finite)}')
    print(f'  > -1.0       : {(pm_r2_finite > -1.0).sum()} / {len(pm_r2_finite)}')
else:
    print('\n=== headline B: no PM_xxxx rows in this dataset ===')

# Top/bottom-fit examples (high-variance subset)
ranked = sorted(top_idx, key=lambda i: -r2[i] if np.isfinite(r2[i]) else 1e9)
print(f'\n5 best-fit high-variance rows:')
for i in ranked[:5]:
    print(f'  {names[i]:30s}  R²={r2[i]:+.3f}  var={var_per_row[i]:.3f}')
print(f'\n5 worst-fit high-variance rows:')
for i in ranked[-5:]:
    print(f'  {names[i]:30s}  R²={r2[i]:+.3f}  var={var_per_row[i]:.3f}')


In [ ]:
import matplotlib.pyplot as plt

# Roll out the predicted dx from x at t=0 and compare to ground truth
x = X[0].copy()
rollout = [x.copy()]
with torch.no_grad():
    for _ in range(T - 1):
        dx = model(torch.from_numpy(x).unsqueeze(0)).squeeze(0).numpy()
        x = x + dx
        rollout.append(x.copy())
ROLL = np.stack(rollout, axis=0)                     # (T, S) in log1p space

# Pick 4 representative high-variance rows
rows_to_plot = ranked[:4]
fig, axs = plt.subplots(2, 2, figsize=(12, 7))
for ax, idx in zip(axs.flat, rows_to_plot):
    t = np.arange(T)
    ax.plot(t, X[:, idx],     label='truth (signed-log1p)',       lw=1)
    ax.plot(t, ROLL[:, idx],  label='rollout',                     lw=1)
    ax.set_title(f'{names[idx]}   R²={r2[idx]:+.3f}', fontsize=10)
    ax.set_xlabel('time (s)')
    ax.legend(fontsize=8)
fig.suptitle('Replicate 50 hold-out: ground truth vs autoregressive rollout',
             y=1.02)
fig.tight_layout()
plt.show()

### Reading the results

- **Median R² over top-100 high-variance rows** is the headline single number. Above 0.5 = the surrogate explains a meaningful chunk of the dynamics on the species that actually move; near 0 = the model is fitting a constant-mean baseline; sharply negative = autoregressive instability or a transform bug.
- **Rollout drift**: the rollout plot is the harder test — even small per-step errors accumulate. If the trained model can't roll out for 7200 steps without going off-manifold, that's expected for a v0 single-step MLP and motivates a v2 with teacher-forcing + a longer-horizon loss.
- **Worst-fit rows** are usually species whose dynamics are dominated by stochastic firing of low-rate reactions; a deterministic mean predictor can't follow them. That's a property of the data, not the model.